In [1]:
# 기본
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 경고 뜨지 않게 설정
import warnings
warnings.filterwarnings('ignore')

# 그래프 설정
sns.set()

# 그래프 기본 설정
# plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['figure.figsize'] = 12, 6
plt.rcParams['font.size'] = 14
plt.rcParams['axes.unicode_minus'] = False

# 데이터 전처리 알고리즘
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

# 학습용과 검증용으로 나누는 함수
from sklearn.model_selection import train_test_split

# 교차 검증
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import cross_validate
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold

# 평가함수
# 분류용
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score

# 회귀용
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error

# 모델의 최적의 하이퍼 파라미터를 찾기 위한 도구
from sklearn.model_selection import GridSearchCV

# 머신러닝 알고리즘 - 분류
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import VotingClassifier

# 머신러닝 알고리즘 - 회귀
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.linear_model import ElasticNet
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import GradientBoostingRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import VotingRegressor


# 학습 모델 저장을 위한 라이브러리
import pickle

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
dfE=pd.read_csv('/content/drive/MyDrive/은서님 파일/model3_E,notE(2).csv')
dfE

,ID,Segment
0,TEST_00000,E
1,TEST_00001,E
2,TEST_00002,E
3,TEST_00003,E
4,TEST_00004,E
...,...,...
99995,TEST_99995,E
99996,TEST_99996,E
99997,TEST_99997,E
99998,TEST_99998,Not_E


In [4]:
dfnotAB=pd.read_csv('/content/drive/MyDrive/은서님 파일/model3_A,B,notAB(2).csv')
dfnotAB

,ID,Segment
0,TEST_00010,notAB
1,TEST_00016,notAB
2,TEST_00032,notAB
3,TEST_00034,notAB
4,TEST_00037,notAB
...,...,...
16698,TEST_99947,notAB
16699,TEST_99957,notAB
16700,TEST_99961,notAB
16701,TEST_99982,notAB


In [5]:
dfCD=pd.read_csv('/content/drive/MyDrive/은서님 파일/model8134_C,D.csv')
dfCD

,ID,Segment
0,TEST_00010,D
1,TEST_00016,D
2,TEST_00032,D
3,TEST_00034,D
4,TEST_00037,C
...,...,...
16683,TEST_99947,D
16684,TEST_99957,D
16685,TEST_99961,D
16686,TEST_99982,C


In [6]:
# 1. dfCD에서 ID와 진짜 Segment만 추출
ab_mapping = dfnotAB[['ID', 'Segment']]

# 2. dfE와 ID 기준으로 병합 (진짜 Segment 값을 붙이기 위해)
dfE_updated = dfE.merge(ab_mapping, on='ID', how='left', suffixes=('', '_true'))

# 3. Not_E인 경우에만 진짜 Segment(C/D)로 교체
dfE_updated['Segment'] = dfE_updated.apply(
    lambda row: row['Segment_true'] if row['Segment'] == 'Not_E' and pd.notnull(row['Segment_true']) else row['Segment'],
    axis=1
)

# 4. 보조 컬럼 제거
dfE_updated = dfE_updated.drop(columns='Segment_true')


In [7]:
dfE_updated['Segment'].value_counts()

,count
Segment,
E,83297
notAB,16688
A,14
B,1


In [8]:
# 1. dfCD에서 ID와 진짜 Segment만 추출
cd_mapping = dfCD[['ID', 'Segment']]

# 2. dfE와 ID 기준으로 병합 (진짜 Segment 값을 붙이기 위해)
dfE_updated2 = dfE_updated.merge(cd_mapping, on='ID', how='left', suffixes=('', '_true'))

# 3. Not_E인 경우에만 진짜 Segment(C/D)로 교체
dfE_updated2['Segment'] = dfE_updated2.apply(
    lambda row: row['Segment_true'] if row['Segment'] == 'notAB' and pd.notnull(row['Segment_true']) else row['Segment'],
    axis=1
)

# 4. 보조 컬럼 제거
dfE_updated2 = dfE_updated2.drop(columns='Segment_true')


In [9]:
dfE_updated2['Segment'].value_counts()

,count
Segment,
E,83297
D,12873
C,3815
A,14
B,1


In [10]:
dfE_updated2

,ID,Segment
0,TEST_00000,E
1,TEST_00001,E
2,TEST_00002,E
3,TEST_00003,E
4,TEST_00004,E
...,...,...
99995,TEST_99995,E
99996,TEST_99996,E
99997,TEST_99997,E
99998,TEST_99998,C


In [11]:
dfE_updated2.to_csv('/content/drive/MyDrive/은서님 파일/result_model3(8134_E수정).csv',index=False, encoding='utf-8-sig')